In [15]:
import fastf1
import requests
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
ROUND_FOR_PREDICTION = 5

In [2]:
schedule = fastf1.get_event_schedule(2025)
schedule

req         WARNING 	DEFAULT CACHE ENABLED! (1.64 GB) C:\Users\grzeg\AppData\Local\Temp\fastf1


,RoundNumber,Country,Location,OfficialEventName,EventDate,EventName,EventFormat,Session1,Session1Date,Session1DateUtc,...,Session3,Session3Date,Session3DateUtc,Session4,Session4Date,Session4DateUtc,Session5,Session5Date,Session5DateUtc,F1ApiSupport
0,0,Bahrain,Sakhir,FORMULA 1 ARAMCO PRE-SEASON TESTING 2025,2025-02-28,Pre-Season Testing,testing,Practice 1,2025-02-26 10:00:00+03:00,2025-02-26 07:00:00,...,Practice 3,2025-02-28 10:00:00+03:00,2025-02-28 07:00:00,None,NaT,NaT,None,NaT,NaT,True
1,1,Australia,Melbourne,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,2025-03-16,Australian Grand Prix,conventional,Practice 1,2025-03-14 12:30:00+11:00,2025-03-14 01:30:00,...,Practice 3,2025-03-15 12:30:00+11:00,2025-03-15 01:30:00,Qualifying,2025-03-15 16:00:00+11:00,2025-03-15 05:00:00,Race,2025-03-16 15:00:00+11:00,2025-03-16 04:00:00,True
2,2,China,Shanghai,FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2025,2025-03-23,Chinese Grand Prix,sprint_qualifying,Practice 1,2025-03-21 11:30:00+08:00,2025-03-21 03:30:00,...,Sprint,2025-03-22 11:00:00+08:00,2025-03-22 03:00:00,Qualifying,2025-03-22 15:00:00+08:00,2025-03-22 07:00:00,Race,2025-03-23 15:00:00+08:00,2025-03-23 07:00:00,True
3,3,Japan,Suzuka,FORMULA 1 LENOVO JAPANESE GRAND PRIX 2025,2025-04-06,Japanese Grand Prix,conventional,Practice 1,2025-04-04 11:30:00+09:00,2025-04-04 02:30:00,...,Practice 3,2025-04-05 11:30:00+09:00,2025-04-05 02:30:00,Qualifying,2025-04-05 15:00:00+09:00,2025-04-05 06:00:00,Race,2025-04-06 14:00:00+09:00,2025-04-06 05:00:00,True
4,4,Bahrain,Sakhir,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2025,2025-04-13,Bahrain Grand Prix,conventional,Practice 1,2025-04-11 14:30:00+03:00,2025-04-11 11:30:00,...,Practice 3,2025-04-12 15:30:00+03:00,2025-04-12 12:30:00,Qualifying,2025-04-12 19:00:00+03:00,2025-04-12 16:00:00,Race,2025-04-13 18:00:00+03:00,2025-04-13 15:00:00,True
5,5,Saudi Arabia,Jeddah,FORMULA 1 STC SAUDI ARABIAN GRAND PRIX 2025,2025-04-20,Saudi Arabian Grand Prix,conventional,Practice 1,2025-04-18 16:30:00+03:00,2025-04-18 13:30:00,...,Practice 3,2025-04-19 16:30:00+03:00,2025-04-19 13:30:00,Qualifying,2025-04-19 20:00:00+03:00,2025-04-19 17:00:00,Race,2025-04-20 20:00:00+03:00,2025-04-20 17:00:00,True
6,6,United States,Miami,FORMULA 1 CRYPTO.COM MIAMI GRAND PRIX 2025,2025-05-04,Miami Grand Prix,sprint_qualifying,Practice 1,2025-05-02 12:30:00-04:00,2025-05-02 16:30:00,...,Sprint,2025-05-03 12:00:00-04:00,2025-05-03 16:00:00,Qualifying,2025-05-03 16:00:00-04:00,2025-05-03 20:00:00,Race,2025-05-04 16:00:00-04:00,2025-05-04 20:00:00,True
7,7,Italy,Imola,FORMULA 1 AWS GRAN PREMIO DEL MADE IN ITALY E ...,2025-05-18,Emilia Romagna Grand Prix,conventional,Practice 1,2025-05-16 13:30:00+02:00,2025-05-16 11:30:00,...,Practice 3,2025-05-17 12:30:00+02:00,2025-05-17 10:30:00,Qualifying,2025-05-17 16:00:00+02:00,2025-05-17 14:00:00,Race,2025-05-18 15:00:00+02:00,2025-05-18 13:00:00,True
8,8,Monaco,Monaco,FORMULA 1 TAG HEUER GRAND PRIX DE MONACO 2025,2025-05-25,Monaco Grand Prix,conventional,Practice 1,2025-05-23 13:30:00+02:00,2025-05-23 11:30:00,...,Practice 3,2025-05-24 12:30:00+02:00,2025-05-24 10:30:00,Qualifying,2025-05-24 16:00:00+02:00,2025-05-24 14:00:00,Race,2025-05-25 15:00:00+02:00,2025-05-25 13:00:00,True
9,9,Spain,Barcelona,FORMULA 1 ARAMCO GRAN PREMIO DE ESPAÑA 2025,2025-06-01,Spanish Grand Prix,conventional,Practice 1,2025-05-30 13:30:00+02:00,2025-05-30 11:30:00,...,Practice 3,2025-05-31 12:30:00+02:00,2025-05-31 10:30:00,Qualifying,2025-05-31 16:00:00+02:00,2025-05-31 14:00:00,Race,2025-06-01 15:00:00+02:00,2025-06-01 13:00:00,True


In [3]:
# Which round comes next
NEXT_GP = 6

def get_sessions(NEXT_GP):
    all_data = []
    for round in range(1, NEXT_GP):
        for session_type in ['FP1', 'FP2', 'FP3', 'Q', 'R']:
            try:
                session = fastf1.get_session(2025, round, session_type)
                session.load()
                all_data.append(session)
    
            except Exception as e:
                print(f"Skipped Round {round} {session_type}: {e}")
    return all_data

In [4]:
all_data = get_sessions(NEXT_GP)

core           INFO 	Loading data for Australian Grand Prix - Practice 1 [v3.5.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '5', '6', '7', '10', '12', '14', '16', '18', '22', '23', '27', '30', '31', '44', '55', '63', '81', '87']
core           INFO 	Loading data for Australian Grand Prix - Practice 2 [v3.5.3]
req            INFO 	Usin

Skipped Round 2 FP2: Session type 'FP2' does not exist for this event
Skipped Round 2 FP3: Session type 'FP3' does not exist for this event


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '63', '4', '1', '44', '16', '6', '12', '22', '23', '31', '27', '14', '18', '55', '10', '87', '7', '5', '30']
core           INFO 	Loading data for Chinese Grand Prix - Race [v3.5.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count


In [53]:
all_data

[2025 Season Round 1: Australian Grand Prix - Practice 1,
 2025 Season Round 1: Australian Grand Prix - Practice 2,
 2025 Season Round 1: Australian Grand Prix - Practice 3,
 2025 Season Round 1: Australian Grand Prix - Qualifying,
 2025 Season Round 1: Australian Grand Prix - Race,
 2025 Season Round 2: Chinese Grand Prix - Practice 1,
 2025 Season Round 2: Chinese Grand Prix - Qualifying,
 2025 Season Round 2: Chinese Grand Prix - Race,
 2025 Season Round 3: Japanese Grand Prix - Practice 1,
 2025 Season Round 3: Japanese Grand Prix - Practice 2,
 2025 Season Round 3: Japanese Grand Prix - Practice 3,
 2025 Season Round 3: Japanese Grand Prix - Qualifying,
 2025 Season Round 3: Japanese Grand Prix - Race,
 2025 Season Round 4: Bahrain Grand Prix - Practice 1,
 2025 Season Round 4: Bahrain Grand Prix - Practice 2,
 2025 Season Round 4: Bahrain Grand Prix - Practice 3,
 2025 Season Round 4: Bahrain Grand Prix - Qualifying,
 2025 Season Round 4: Bahrain Grand Prix - Race,
 2025 Season R

In [54]:
all_data[4].results.columns

Index(['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName',
       'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName',
       'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition',
       'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points'],
      dtype='object')

In [57]:
all_data[7].results[['DriverNumber', 'TeamName','Points']]

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,CountryCode,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points
81,81,O PIASTRI,PIA,piastri,McLaren,FF8000,mclaren,Oscar,Piastri,Oscar Piastri,...,,1.0,1,1.0,NaT,NaT,NaT,0 days 01:30:55.026000,Finished,25.0
4,4,L NORRIS,NOR,norris,McLaren,FF8000,mclaren,Lando,Norris,Lando Norris,...,,2.0,2,3.0,NaT,NaT,NaT,0 days 00:00:09.748000,Finished,18.0
63,63,G RUSSELL,RUS,russell,Mercedes,27F4D2,mercedes,George,Russell,George Russell,...,,3.0,3,2.0,NaT,NaT,NaT,0 days 00:00:11.097000,Finished,15.0
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,...,,4.0,4,4.0,NaT,NaT,NaT,0 days 00:00:16.656000,Finished,12.0
31,31,E OCON,OCO,ocon,Haas F1 Team,B6BABD,haas,Esteban,Ocon,Esteban Ocon,...,,5.0,5,11.0,NaT,NaT,NaT,0 days 00:00:49.969000,Finished,10.0
12,12,K ANTONELLI,ANT,antonelli,Mercedes,27F4D2,mercedes,Kimi,Antonelli,Kimi Antonelli,...,,6.0,6,8.0,NaT,NaT,NaT,0 days 00:00:53.748000,Finished,8.0
23,23,A ALBON,ALB,albon,Williams,64C4FF,williams,Alexander,Albon,Alexander Albon,...,,7.0,7,10.0,NaT,NaT,NaT,0 days 00:00:56.321000,Finished,6.0
87,87,O BEARMAN,BEA,bearman,Haas F1 Team,B6BABD,haas,Oliver,Bearman,Oliver Bearman,...,,8.0,8,17.0,NaT,NaT,NaT,0 days 00:01:01.303000,Finished,4.0
18,18,L STROLL,STR,stroll,Aston Martin,229971,aston_martin,Lance,Stroll,Lance Stroll,...,,9.0,9,14.0,NaT,NaT,NaT,0 days 00:01:10.204000,Finished,2.0
55,55,C SAINZ,SAI,sainz,Williams,64C4FF,williams,Carlos,Sainz,Carlos Sainz,...,,10.0,10,15.0,NaT,NaT,NaT,0 days 00:01:16.387000,Finished,1.0


In [78]:
def create_df(all_data):
    """
    Create a DataFrame with F1 lap times and cumulative points for each driver per round.

    This function processes a list of F1 session data to compile a DataFrame containing the best lap times from free practice sessions (FP1, FP2, FP3), qualifying sessions (Q1, Q2, Q3), the fastest race lap, and accumulated championship points for each driver in each race round. It iterates through each session, extracting and aggregating relevant data, ensuring each round is processed systematically. For each round, `Total_Points` is initialized with the cumulative points from all previous rounds for each driver, then updated with points from the current race.

    Parameters:
        all_data (list): A list of session objects (e.g., from FastF1), where each object corresponds to a specific session (Practice 1, Practice 2, Practice 3, Qualifying, or Race). Each session object must have attributes like `event`, `session_info`, `name`, `laps`, and `results`.

    Returns:
        pandas.DataFrame: A DataFrame with columns including:
            - 'Round': Race round number.
            - 'Event': Official name of the event.
            - 'Driver': Driver's name.
            - 'DriverNumber': Driver's number.
            - 'Team': Driver's team.
            - 'FP1_LapTime', 'FP1_Compound', 'FP1_SpeedFL', 'FP1_SpeedST': Best lap time and related data from Practice 1.
            - 'FP2_LapTime', 'FP2_Compound', 'FP2_SpeedFL', 'FP2_SpeedST': Best lap time and related data from Practice 2.
            - 'FP3_LapTime', 'FP3_Compound', 'FP3_SpeedFL', 'FP3_SpeedST': Best lap time and related data from Practice 3.
            - 'Q1', 'Q2', 'Q3': Lap times from qualifying sessions.
            - 'FastestRaceLap': Fastest lap time during the race.
            - 'Total_Points': Cumulative championship points for the driver up to the current round.

    Notes:
        - Assumes `all_data` contains session objects in a logical order (e.g., Practice 1 before Practice 2) with consistent attributes.
        - Time columns are converted to seconds for consistency using `total_seconds()`.
        - `Total_Points` is initialized to 0.0 for a driver’s first round (or if no previous data exists) and set to the cumulative total from previous rounds for subsequent rounds. Points from each race are then added to update the total.
        - Handles edge cases such as new drivers, missing rounds, or empty DataFrames (e.g., during the first round) to prevent errors when accessing columns like 'DriverNumber'.
        - The function uses pandas for data manipulation and assumes the input data is compatible with FastF1 or similar F1 data APIs.
        """
    results = pd.DataFrame()
    rounds = []
    
    for rnd in range(len(all_data)):
        rnd_no = int(all_data[rnd].event.RoundNumber)
        session_info = all_data[rnd].session_info['Meeting']['OfficialName']
        session_name = all_data[rnd].name
        initial_total_points = 0.0
    
        if rnd_no not in rounds:
            
            if session_name == 'Practice 1':
                
                data = all_data[rnd].laps[['Driver', 'DriverNumber', 'Team', 'LapTime', 'Compound', 'SpeedFL', 'SpeedST']].copy()
                data = data.sort_values('LapTime', ascending=True)
                data = data.drop_duplicates(subset='Driver', keep='first')
                row_data_list = []
                for _, row in data.iterrows():
                    driver_number = row['DriverNumber']
                    # Find the cumulative Total_Points from previous rounds for this driver
                    if not results.empty and 'DriverNumber' in results.columns:
                        previous_rounds = results[(results['DriverNumber'] == driver_number) & (results['Round'] < rnd_no)]
                        if not previous_rounds.empty:
                            initial_total_points = previous_rounds['Total_Points'].max()  # Take the max (cumulative) from previous rounds
                        else:
                            initial_total_points = 0.0
                        
                    row_data_list.append({
                        'Round': rnd_no,
                        'Event': session_info,
                        'Driver': row['Driver'],
                        'DriverNumber': row['DriverNumber'],
                        'Team': row['Team'],
                        'FP1_LapTime': row['LapTime'],
                        'FP1_Compound': row['Compound'],
                        'FP1_SpeedFL': row['SpeedFL'],
                        'FP1_SpeedST': row['SpeedST'],
                        'Total_Points' : initial_total_points,
                        'Position' : int(0)
                    })
                results = pd.concat([results, pd.DataFrame(row_data_list)], ignore_index=True)
                    
            if session_name =='Practice 2':
                
                data = all_data[rnd].laps[['Driver', 'DriverNumber', 'Team', 'LapTime', 'Compound', 'SpeedFL', 'SpeedST']].copy()
                data = data.sort_values('LapTime', ascending=True)
                data = data.drop_duplicates(subset='Driver', keep='first')
                for _, row in data.iterrows():
                    results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), 
                               ['FP2_LapTime', 'FP2_Compound', 'FP2_SpeedFL', 'FP2_SpeedST']] = [
                                   row['LapTime'], row['Compound'], row['SpeedFL'], row['SpeedST']
                               ]
            if session_name =='Practice 3':
                
                data = all_data[rnd].laps[['Driver', 'DriverNumber', 'Team', 'LapTime', 'Compound', 'SpeedFL', 'SpeedST']].copy()
                data = data.sort_values('LapTime', ascending=True)
                data = data.drop_duplicates(subset='Driver', keep='first')
                for _, row in data.iterrows():
                    results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), 
                               ['FP3_LapTime', 'FP3_Compound', 'FP3_SpeedFL', 'FP3_SpeedST']] = [
                                   row['LapTime'], row['Compound'], row['SpeedFL'], row['SpeedST']
                               ]                   
        
            if session_name =='Qualifying':
        
                data = all_data[rnd].results[['DriverNumber','Q1','Q2','Q3']].copy()
                data = data.sort_values('Q3')
                for _, row in data.iterrows():
                    results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), ['Q1','Q2','Q3']] = [row['Q1'], row['Q2'], row['Q3']]
            
            if session_name =='Race':
                data = all_data[rnd].laps[['DriverNumber','LapTime']].copy()
                data = data.sort_values('LapTime', ascending=True)
                data = data.drop_duplicates(subset='DriverNumber', keep='first')
                points = all_data[rnd].results[['DriverNumber','Points', 'Position']]
    
                    
                for _, row in data.iterrows():
                    results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), 'FastestRaceLap'] = row['LapTime']
        
                for _, row in points.iterrows():
                    results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), 
                        'Total_Points'] += row['Points']
                    results.loc[(results['DriverNumber'] == row['DriverNumber']) & (results['Round'] == rnd_no), 
                        'Position'] += row['Position']
                rounds.append(rnd_no)
                
    time_columns = ['FP1_LapTime', 'FP2_LapTime','FP3_LapTime','Q1','Q2','Q3', 'FastestRaceLap']
    
    for col in time_columns:
        results[col] = results[col].apply(lambda x: x.total_seconds() if pd.notna(x) else np.nan)
        
    results['Total_Points'] = results['Total_Points'].fillna(0)

    return results

In [79]:
results = create_df(all_data)

In [83]:
results.query("Round == 5")[['Round','Driver','FP2_LapTime', 'FP3_LapTime', 'Q1','Q2','Q3','FastestRaceLap', 'Total_Points', 'Position']].sort_values('Position')


,Round,Driver,FP2_LapTime,FP3_LapTime,Q1,Q2,Q3,FastestRaceLap,Total_Points,Position
83,5,PIA,88.430,87.513,87.901,87.545,87.304,92.228,92.0,1
88,5,VER,88.547,88.334,87.778,87.529,87.294,92.280,73.0,2
82,5,LEC,88.749,88.372,88.552,87.866,87.670,92.192,31.0,3
81,5,NOR,88.267,87.489,87.805,87.481,NaN,91.778,88.0,4
85,5,RUS,88.973,88.116,88.282,87.599,87.407,92.893,50.0,5
92,5,ANT,89.242,88.679,88.128,87.798,87.866,92.396,36.0,6
87,5,HAM,89.371,88.780,88.372,88.102,88.201,92.600,23.0,7
86,5,SAI,88.942,88.570,88.354,88.024,88.164,92.466,5.0,8
84,5,ALB,89.220,88.389,88.279,88.109,NaN,93.477,20.0,9
94,5,HAD,89.306,88.769,88.571,88.418,NaN,93.257,5.0,10


In [94]:
# check the correlation between lap times:
results[['FP1_LapTime', 'FP2_LapTime', 'FP3_LapTime', 'Q1','Q2','Q3','FastestRaceLap']].corr()

,FP1_LapTime,FP2_LapTime,FP3_LapTime,Q1,Q2,Q3,FastestRaceLap
FP1_LapTime,1.000000,0.983521,0.872701,0.989646,0.987329,0.984063,0.510978
FP2_LapTime,0.983521,1.000000,0.869931,0.994791,0.993640,0.994267,0.504504
FP3_LapTime,0.872701,0.869931,1.000000,0.874269,0.875935,0.873718,0.431291
Q1,0.989646,0.994791,0.874269,1.000000,0.998568,0.997108,0.506710
Q2,0.987329,0.993640,0.875935,0.998568,1.000000,0.998247,0.506705
Q3,0.984063,0.994267,0.873718,0.997108,0.998247,1.000000,0.510647
FastestRaceLap,0.510978,0.504504,0.431291,0.506710,0.506705,0.510647,1.000000


In [97]:
def handle_nan_times(df):
    """
    Impute missing lap time values in a Formula 1 dataset for specified sessions and fastest race lap.

    This function fills NaN values in the following columns for each driver in each race round:
    - 'FP1_LapTime', 'FP2_LapTime', 'FP3_LapTime', 'Q1', 'Q2', 'Q3': Lap times for free practice and qualifying sessions.
    - 'FastestRaceLap': The fastest lap time during the race.

    For 'FastestRaceLap', missing values are filled with the median 'FastestRaceLap' of that round.

    For each of the session lap times ('FP1_LapTime' to 'Q3'), if a value is missing for a driver in a round:
    - If the driver has any other non-missing lap times in other sessions of the same round, the missing value is filled with the minimum (best) of those available lap times.
    - If the driver has no other non-missing lap times in that round, the missing value is filled with the median lap time for that specific session in that round across all drivers.

    A small random noise (uniform between -0.5 and 0.5) is added to the imputed lap times for the session columns to introduce variability.

    Parameters:
        df (pandas.DataFrame): Input DataFrame containing F1 data with columns:
            - 'Round': Race round number.
            - 'DriverNumber': Driver's number.
            - 'FP1_LapTime', 'FP2_LapTime', 'FP3_LapTime', 'Q1', 'Q2', 'Q3': Lap times for respective sessions.
            - 'FastestRaceLap': Fastest lap time during the race.

    Returns:
        pandas.DataFrame: The input DataFrame with NaN values in the specified columns filled.

    Notes:
        - The function modifies the input DataFrame in-place.
        - Assumes that the DataFrame has the necessary columns and that lap times are numeric.
        - The imputation logic uses the best time from any session, which may not preserve the temporal order of sessions (e.g., using a Q3 time to fill FP2).
        - Requires the numpy library for handling NaN checks and random noise generation.
    """
    columns = ['FP1_LapTime', 'FP2_LapTime', 'FP3_LapTime','Q1','Q2','Q3']
    
    for _, group in df.groupby(['Round', 'DriverNumber']):
        mask = (df['Round'] == group['Round'].iloc[0]) & (df['DriverNumber'] == group['DriverNumber'].iloc[0])
        round_mask = df['Round'] == group['Round'].iloc[0]
        valid_times = group[columns].values.flatten()
        valid_times = valid_times[~np.isnan(valid_times)]
        fill_value_col = min(valid_times) if valid_times.size > 0 else df.loc[round_mask, columns].max().max()
        df.loc[(mask & df['FastestRaceLap'].isna()), 'FastestRaceLap'] = df.loc[round_mask,'FastestRaceLap'].median()
        
        for i, col in enumerate(columns):
            # If the driver hasn't participated in this session, fill with his best available time or the worse of others
            if np.isnan(group[col].iloc[0]):
                if valid_times.size > 0:
                    # Fill with the driver's best time so far
                    fill_value_col = min(valid_times)
                else:
                    # If no valid times yet for the driver, fill with the lowest time of other drivers in the same session
            
                    fill_value_col = df.loc[round_mask, col].median()
                    
                # Add some noise (optional)
                fill_value_col += np.random.uniform(-0.5, 0.5)
            
                # Fill the NaN value for the current session
                df.loc[mask & results[col].isna(), col] = fill_value_col
                              
    return df

In [98]:
results = handle_nan_times(results)

In [99]:
results.query("Round == 2")[['Round','Driver','FP2_LapTime', 'FP3_LapTime', 'Q1','Q2','Q3','FastestRaceLap', 'Total_Points']]

,Round,Driver,FP2_LapTime,FP3_LapTime,Q1,Q2,Q3,FastestRaceLap,Total_Points
20,2,NOR,90.907178,90.412647,90.983,90.787000,90.793000,95.454,43.0
21,2,LEC,90.930469,90.526052,91.579,91.450000,91.021000,96.157,4.0
22,2,PIA,91.085730,90.831050,91.591,91.200000,90.641000,95.520,27.0
23,2,HAM,90.978966,91.322094,91.690,91.501000,90.927000,95.069,1.0
24,2,RUS,90.638786,90.692970,91.295,91.307000,90.723000,95.816,30.0
25,2,HUL,91.773747,91.195157,91.921,91.632000,91.518914,97.275,6.0
26,2,ALB,91.181444,91.343684,91.503,91.595000,91.706000,96.254,16.0
27,2,ALO,92.005865,91.336304,91.719,91.688000,91.933248,99.256,0.0
28,2,ANT,91.483266,90.931592,91.676,91.590000,91.103000,96.046,20.0
29,2,TSU,91.426058,91.475992,91.238,91.260000,91.638000,95.871,0.0


In [100]:
#Training

data = results.query("Round > 0 and Round < @ROUND_FOR_PREDICTION")[['FP1_LapTime', 'FP2_LapTime', 'FP3_LapTime', 'Q1', 'Q2', 'Q3', 'FastestRaceLap']].copy()

columns_x = ['FP1_LapTime', 'FP2_LapTime', 'FP3_LapTime', 'Q1', 'Q2', 'Q3','FastestRaceLap']
models = {}
X_train = []

for i in range(1, len(columns_x)):
    X_train = data[columns_x[:i]]
    y_train = data[columns_x[i]]
    model = LinearRegression().fit(X_train, y_train)
    models[columns_x[i]] = model

for name, model in models.items():
    print(f"Weights for {name}: {model.coef_}")  # For FP2_LapTime, FP3_LapTime
    print(f"Intercept for {name}: {model.intercept_}")

Weights for FP2_LapTime: [0.93993996]
Intercept for FP2_LapTime: 4.211694521810969
Weights for FP3_LapTime: [0.50730564 0.44534968]
Intercept for FP3_LapTime: 4.064118966853897
Weights for Q1: [0.3104586  0.7164808  0.00748714]
Intercept for Q1: -3.7443736815880584
Weights for Q2: [-0.05931527  0.03934278  0.01187664  1.00987828]
Intercept for Q2: -0.23813125746426067
Weights for Q3: [-0.08950073  0.17600236  0.00107284  0.07252444  0.85303712]
Intercept for Q3: -1.2427596683290858
Weights for FastestRaceLap: [ 0.72684472 -0.86063695 -0.08442839 -0.580642    0.43733955  1.03969877]
Intercept for FastestRaceLap: 32.87497657039852


### Predict each time after each session results starting from FP1.

In [101]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
"""
Impute missing values in F1 data using sequential prediction models.

This function takes a DataFrame of F1 data, a round number for prediction, and a dictionary of pre-trained models.
It creates a subset of the data up to the specified round, then sequentially imputes missing values for each feature
(from 'FP2_LapTime' to 'FastestRaceLap') using the previous features as predictors.

For each feature to be imputed, it uses a pre-trained model to predict the missing values based on the available
(or previously imputed) values of the preceding features. The function also collects actual and predicted values
for evaluation and creates new columns for predicted values and their differences (deltas).

Parameters:
    results (pandas.DataFrame): The F1 data containing columns 'Round', 'Driver', 'FP1_LapTime', 'FP2_LapTime',
                                'FP3_LapTime', 'Q1', 'Q2', 'Q3', 'FastestRaceLap', 'Total_Points'.
    ROUND_FOR_PREDICTION (int): The round number up to which to include data for imputation.
    models (dict): A dictionary where keys are the column names to be imputed ('FP2_LapTime', 'FP3_LapTime', 'Q1',
                   'Q2', 'Q3', 'FastestRaceLap'), and values are trained sklearn models that predict each column
                   based on the preceding columns in ['FP1_LapTime', 'FP2_LapTime', 'FP3_LapTime', 'Q1', 'Q2', 'Q3'].

Returns:
    list of pandas.DataFrame: A list where each element is a DataFrame with the original columns plus
                              '{column}_Predicted' and '{column}_Delta' for each imputed column.

Notes:
    - The first column ('FP1_LapTime') is not imputed and must not contain missing values.
    - The models must be pre-trained and compatible with the input features.
    - Each DataFrame in the returned list represents the state of the data after imputing a specific column.
"""

data_pred = results.query("Round <= @ROUND_FOR_PREDICTION")[['Round','Driver','FP1_LapTime', 'FP2_LapTime', 'FP3_LapTime', 'Q1', 'Q2', 'Q3', 'FastestRaceLap', 'Total_Points']].copy()
columns_x = ['FP1_LapTime', 'FP2_LapTime', 'FP3_LapTime', 'Q1', 'Q2', 'Q3','FastestRaceLap']

# Initialize dictionaries to store the actuals and predictions for each model
actuals = {col: [] for col in columns_x}
predictions = {col: [] for col in columns_x}
all_predictions = []

# Perform predictions and collect the actual and predicted values
for i in range(1, len(columns_x)):
    mask = data_pred[columns_x[i]].isna()
    X_input = data_pred.loc[:, columns_x[:i]]
    y_pred = models[columns_x[i]].predict(X_input)
    
    # For each column, store the actual and predicted values
    actuals[columns_x[i]].extend(data_pred[columns_x[i]].dropna())  # Add the actual values (without NaN)
    predictions[columns_x[i]].extend(y_pred)  # Add the predicted values
    
    # Impute NaN values with predictions
    data_pred.loc[mask, columns_x[i]] = y_pred[mask]
    data_post_pred = data_pred.copy()
    data_post_pred[f'{columns_x[i]}_Predicted'] = y_pred
    data_post_pred[f'{columns_x[i]}_Delta'] = data_post_pred[columns_x[i]] - y_pred

    all_predictions.append(data_post_pred)
    

"""
How to Use F1 Lap Time Predictions

`all_predictions` is a list with 6 predicted lap times for a driver in a race round, starting from Free Practice 2 (FP2) to the Fastest Race Lap. It starts with FP1 time from `results` and builds predictions step-by-step, where each prediction uses FP1 plus all earlier predictions. Use these to guess how fast a driver will go in practice, qualifying, or the race.

What's Included:
    - 0: FP2_LapTime - Predicted FP2 time, using only FP1_LapTime from `results`.
    - 1: FP3_LapTime - Predicted FP3 time, using FP1_LapTime and predicted FP2_LapTime.
    - 2: Q1 - Predicted Qualifying 1 time, using FP1_LapTime, predicted FP2_LapTime, and FP3_LapTime.
    - 3: Q2 - Predicted Qualifying 2 time, using FP1_LapTime, predicted FP2_LapTime, FP3_LapTime, and Q1.
    - 4: Q3 - Predicted Qualifying 3 time, using FP1_LapTime, predicted FP2_LapTime, FP3_LapTime, Q1, and Q2.
    - 5: FastestRaceLap - Predicted fastest race lap, using FP1_LapTime, predicted FP2_LapTime, FP3_LapTime, Q1, Q2, and Q3.

How to Use It:
When new session starts and we get only FP1 times (First practice session) we can predict all times base on thatone practice session: 
Just set the ROUND_FOR_PREDICTION global variable at the top, and run the script. You can acces the prediction by: all_predictions[0].
Then after next practice lap times FP2 are updated we redownload them (rerun whole script) and enter all_predictions[1]
This will use both known times FP1 and FP2 for prediction of FP3, as well as all other times.
The predictions will get more accurate with more acctual lap times.

    1. **Get a Prediction**:
        - Grab a time with the index: `all_predictions[0]` for FP2, `all_predictions[5]` for race lap, etc.
        - Example: `print(all_predictions[0])` shows the predicted FP2 time in seconds.

    2. **Add to Your Data**:
        - Put predictions into your `results` DataFrame (from `create_df`) to compare with real times or fill gaps.
        - Example:
            ```python
            results.loc[(results['Round'] == 1) & (results['DriverNumber'] == 44), 'FP2_LapTime'] = all_predictions[0]
            ```

    3. **What You Can Do**:
        - See if predictions match real lap times.
        - Guess where drivers will start the race using Q1, Q2, Q3 predictions.
        - Plan race strategies with the FastestRaceLap prediction.

Parameters:
    all_predictions (list): A list of 6 numbers (in seconds) for predicted FP2, FP3, Q1, Q2, Q3, and FastestRaceLap times.

Tips:
    - FP2 prediction only uses FP1 time. Then, FP3 uses FP1 and predicted FP2, Q1 uses FP1, FP2, FP3, and so on, building up to the race lap prediction.
    - Check that FP1_LapTime in `results` is correct, since all predictions start from it.
    - Times are in seconds, just like in `results`.
    - If a driver’s predictions look off, double-check FP1 data or run `handle_nan_times`.

    Examples below.
"""

In [110]:
# The prediction of FP2 using only FP1 time, the rest is either base on that or if all times are available its based on acctual.
all_predictions[0].query("Round == @ROUND_FOR_PREDICTION")

,Round,Driver,FP1_LapTime,FP2_LapTime,FP3_LapTime,Q1,Q2,Q3,FastestRaceLap,Total_Points,FP2_LapTime_Predicted,FP2_LapTime_Delta
80,5,GAS,89.239,89.106000,88.625,88.421,88.025000,88.367000,92.998,6.0,88.090997,1.015003
81,5,NOR,89.246,88.267000,87.489,87.805,87.481000,87.794948,91.778,88.0,88.097576,0.169424
82,5,LEC,89.309,88.749000,88.372,88.552,87.866000,87.670000,92.192,31.0,88.156793,0.592207
83,5,PIA,89.341,88.430000,87.513,87.901,87.545000,87.304000,92.228,92.0,88.186871,0.243129
84,5,ALB,89.606,89.220000,88.389,88.279,88.109000,88.436643,93.477,20.0,88.435955,0.784045
85,5,RUS,89.618,88.973000,88.116,88.282,87.599000,87.407000,92.893,50.0,88.447234,0.525766
86,5,SAI,89.779,88.942000,88.570,88.354,88.024000,88.164000,92.466,5.0,88.598564,0.343436
87,5,HAM,89.815,89.371000,88.780,88.372,88.102000,88.201000,92.600,23.0,88.632402,0.738598
88,5,VER,89.818,88.547000,88.334,87.778,87.529000,87.294000,92.280,73.0,88.635222,-0.088222
89,5,TSU,89.821,88.963000,88.670,88.226,87.990000,88.204000,165.662,2.0,88.638042,0.324958


In [107]:
# all_predictions[4] access the predictions for Q3.
all_predictions[4].query("Round == @ROUND_FOR_PREDICTION").sort_values('Q3')

,Round,Driver,FP1_LapTime,FP2_LapTime,FP3_LapTime,Q1,Q2,Q3,FastestRaceLap,Total_Points,Q3_Predicted,Q3_Delta
88,5,VER,89.818,88.547000,88.334,87.778,87.529000,87.294000,92.280,73.0,87.429249,-0.135249
83,5,PIA,89.341,88.430000,87.513,87.901,87.545000,87.304000,92.228,92.0,87.473037,-0.169037
85,5,RUS,89.618,88.973000,88.116,88.282,87.599000,87.407000,92.893,50.0,87.618157,-0.211157
82,5,LEC,89.309,88.749000,88.372,88.552,87.866000,87.670000,92.192,31.0,87.854006,-0.184006
81,5,NOR,89.246,88.267000,87.489,87.805,87.481000,87.794948,91.778,88.0,87.391269,0.403679
92,5,ANT,89.934,89.242000,88.679,88.128,87.798000,87.866000,92.396,36.0,87.796409,0.069591
93,5,ALO,89.976,89.662000,88.888,88.548,88.303000,87.990010,93.009,0.0,88.328040,-0.338030
86,5,SAI,89.779,88.942000,88.570,88.354,88.024000,88.164000,92.466,5.0,87.966541,0.197459
90,5,LAW,89.907,89.488000,88.861,88.561,88.191000,88.176233,92.998,0.0,88.208964,-0.032731
87,5,HAM,89.815,89.371000,88.780,88.372,88.102000,88.201000,92.600,23.0,88.106892,0.094108
